# Resampling and CV - How to Compare Models Without Fooling Yourself

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/08_cross_validation_model_comparison_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Write the $k$-fold cross-validation estimator for both regression (MSE) and classification (misclassification / score-based) problems
2. Run 5-fold CV on a regression business case (California Housing) and on a classification business case (Breast Cancer), reporting the mean, standard deviation, and 95% confidence interval every time
3. Plot per-fold CV scores with the mean and CI, and compare the CV estimate against the single validation-set score using a second bar plot
4. Interpret whether a single validation score was lucky, unlucky, or representative based on whether it falls inside the CV 95% CI
5. Use the 95% CI overlap rule to decide whether one model (or hyperparameter choice) is **convincingly** better than another on the same task

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Please complete them before submitting your notebook.

---

## 💼 Why This Matters: One Lucky Split or Genuinely Better?

You have two stakeholders pushing back this week — one from each arc of the course so far.

**HomeValue Analytics (California Housing — regression).** Your Ridge pipeline posts a validation $R^2$ of roughly **0.60**. The CFO asks: *"If we had drawn a different 20% for the validation set, would the number have come out the same?"* You cannot answer that with one split.

**State Health Department (Breast Cancer — classification).** Your logistic regression posts a validation ROC-AUC of roughly **0.99**. The chief medical officer asks: *"Is that 0.99 real, or did we just get lucky with the 114 patients in the validation set?"* Again, one split cannot answer.

**Same question in two domains, same solution: cross-validation.** Instead of one 60/20/20 split, run the experiment $k$ times on $k$ different partitions of the training data, then look at the **distribution** of scores — mean, standard deviation, confidence interval — not a single number. A point estimate ± confidence interval is what the CFO and the chief medical officer can actually make a deployment decision with.

> **Today's focus:** Run $k$-fold CV on both business cases, report the mean with a 95% confidence interval, and compare the CV estimate against the single validation-set score to decide whether the single split was lucky, unlucky, or representative.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.metrics import roc_auc_score, mean_squared_error
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
print("✓ Setup complete!")

**Reading the output:**

The setup cell imports the full resampling toolkit used in the rest of the notebook:

- **`KFold`** and **`StratifiedKFold`** — the two CV splitter objects. `StratifiedKFold` preserves the positive-class prevalence in each fold, which is mandatory for classification.
- **`cross_val_score`** and **`cross_validate`** — the convenience functions that handle the train-evaluate loop internally.
- **`train_test_split`** — for the single 60/20/20 split we compare against.
- **`scipy.stats`** — used to compute the Student's $t$ critical value for the 95% confidence interval.
- **`LinearRegression`, `Ridge`, `LogisticRegression`** — the three models used across Sections 2, 3, and the Exercise.

`RANDOM_SEED = 474` is carried forward from NB01–NB07 so every split and CV fold is directly comparable to prior notebooks.

---

## 1. Why Cross-Validation Exists

A single 60/20/20 split gives you one number for validation performance — but that number is **one draw from a distribution**. Which 20% of the data ended up in the validation set is a random choice, and a different random choice would give a different score. With 114 validation patients (breast cancer) or \~4,128 validation tracts (California Housing), the variation from one split to the next can be large enough to flip the ordering of two candidate models.

### $k$-fold CV — the mechanism (watch it happen)

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/KfoldCV.gif" width="640" alt="Animated visualization of 5-fold cross-validation">
</center>

Every frame of the animation is one iteration of $k$-fold CV. The training data is sliced into $k$ non-overlapping **folds**; at each iteration exactly one fold is held out for scoring (the colored stripe) and the other $k-1$ folds are used for fitting the model. The held-out fold rotates through all $k$ positions, so by the end of the cycle **every sample in the training set has been predicted exactly once, by a model that never saw it during fitting**. That is the whole trick — a single pass of the animation gives you $k$ honest validation scores instead of one.

Three properties are worth naming while you watch:

- **Fold size is constant.** Each fold is roughly $n/k$ rows; with $n \approx 13{,}000$ tracts and $k=5$ each fold holds \~2,600 rows.
- **The test set is never touched.** Everything you see rotating belongs to the *training* portion of the 60/20/20 split. The 20% test set stays locked until NB14 or the final project.
- **The $k$ scores are not independent.** Adjacent iterations share $k-2$ of their $k-1$ training folds, so fold scores are correlated. This matters when you build a confidence interval — Student's $t$ with $k-1$ degrees of freedom is the conservative choice for exactly this reason.

### The $k$-fold CV estimator

Formally, partition the training data into $k$ equally-sized folds — call them $C_1, \ldots, C_k$. For each fold $i$, train on the other $k-1$ folds and evaluate on the held-out fold $C_i$. Average the $k$ scores.

**For regression** (ISLP eq. 5.3):

$$\text{CV}_{(k)} \;=\; \frac{1}{k} \sum_{i=1}^{k} \text{MSE}_i
\qquad \text{where} \qquad
\text{MSE}_i \;=\; \frac{1}{\lvert C_i \rvert} \sum_{j \in C_i} \left( y_j - \hat{y}_j^{(-i)} \right)^2$$

$\hat{y}_j^{(-i)}$ is the prediction for sample $j$ from the model trained on the $k-1$ folds that do **not** contain $j$ — the same idea the animation just showed, written as an equation.

**For classification** (ISLP eq. 5.4):

$$\text{CV}_{(k)} \;=\; \frac{1}{k} \sum_{i=1}^{k} \text{Err}_i
\qquad \text{where} \qquad
\text{Err}_i \;=\; \frac{1}{\lvert C_i \rvert} \sum_{j \in C_i} \mathbb{1}\!\left( y_j \neq \hat{y}_j^{(-i)} \right)$$

The structure is identical — only the per-fold loss changes (squared error for regression, misclassification rate for classification). Score-based metrics like $R^2$, ROC-AUC, accuracy, and F1 are monotone transforms of these losses and plug into the same formula:

$$\text{CV}_{(k)}^{\text{score}} \;=\; \frac{1}{k} \sum_{i=1}^{k} \text{score}_i$$

where $\text{score}_i$ is whatever metric is passed to `scoring=` in `cross_val_score` (e.g., `'r2'`, `'roc_auc'`, `'f1'`).

### The limiting case — Leave-One-Out CV (LOOCV)

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/LOOCV.gif" width="640" alt="Animated visualization of Leave-One-Out cross-validation">
</center>

Push $k$ all the way up to $k = n$ and every fold is a single row. That special case is **Leave-One-Out CV (LOOCV)**: train on $n-1$ samples, predict the one that was left out, repeat $n$ times. The animation shows every row taking a turn as the held-out sample.

Two properties fall out of the $k = n$ setting, and they explain why LOOCV is *not* the default in this course:

- **Low bias, because each training fold uses nearly all the data** — the model fit on $n-1$ rows is almost the model you would deploy. This is why LOOCV is unbiased as an estimator of test error.
- **High variance and high compute, because the $n$ training sets are nearly identical** — the $n$ held-out errors are highly correlated, which inflates the variance of the LOOCV estimate. And training $n$ models (\~13,000 on California Housing) is several orders of magnitude slower than training 5 or 10.

The practical default of **$k=5$ or $k=10$** is a deliberate compromise: a little more bias than LOOCV, visibly lower variance in the mean estimator, and compute that finishes while you are still thinking about the result. The rest of this notebook uses $k=5$.

### Summary statistics — always report all three

Running $k$-fold CV gives you $k$ scores $s_1, \ldots, s_k$. Three numbers summarize them:

**Mean** (the point estimate):

$$\bar{s} \;=\; \frac{1}{k} \sum_{i=1}^{k} s_i$$

**Standard deviation** across folds (sample SD with $k-1$ degrees of freedom):

$$\text{SD} \;=\; \sqrt{\frac{1}{k-1} \sum_{i=1}^{k} \bigl(s_i - \bar{s}\bigr)^2}$$

**95% confidence interval** for the true mean, using Student's $t$ with $k-1$ degrees of freedom:

$$\text{CI}_{95\%} \;=\; \bar{s} \;\pm\; t_{0.975,\, k-1} \cdot \frac{\text{SD}}{\sqrt{k}}$$

At $k=5$, $t_{0.975,\, 4} \approx 2.776$, so the half-width of the CI is roughly $1.24 \cdot \text{SD}$. The rest of the notebook always reports all three — mean, SD, and CI — plus a per-fold bar plot. **Never quote a CV mean without the spread.**

---

## 2. K-Fold CV for Regression — California Housing (HomeValue Analytics)

Back to the pricing model from NB01–NB05. HomeValue Analytics has a Ridge pipeline (StandardScaler + Ridge, $\alpha = 1.0$) that posts a validation $R^2 \approx 0.60$ on a single 60/20/20 split. The CFO wants a number with a confidence interval, not a point estimate.

The plan is the standard four-step recipe that repeats in Section 3:

1. Apply the usual 60/20/20 split with `RANDOM_SEED = 474`.
2. Fit on the training set, evaluate once on the validation set — record that **single-split $R^2$**.
3. Run **5-fold CV on the training set only** (the test set stays locked), compute mean, SD, and the 95% CI from the Student's $t$ formula in Section 1.
4. Compare the two side by side in a bar plot and interpret the result.

Plugging $R^2$ into the CV estimator:

$$\text{CV}_{(5)}^{R^2} \;=\; \frac{1}{5} \sum_{i=1}^{5} R^2_i$$

where $R^2_i$ is the coefficient of determination on held-out fold $i$.

> 💡 **Gemini Prompt:** "Load the California Housing dataset. Do a 60/20/20 train/val/test split with `random_state=RANDOM_SEED`. Build a Ridge pipeline (StandardScaler + Ridge with alpha=1.0), fit it on `X_train`, and compute the single-split validation R² on `X_val`. Then run 5-fold `KFold` cross-validation on the training set with `scoring='r2'`. From the fold scores, compute the mean, the sample standard deviation (`ddof=1`), and a 95% confidence interval using `scipy.stats.t.ppf(0.975, df=k-1)`. Print all three numbers plus the fold scores. Render a 1x2 matplotlib figure: left panel, per-fold bar plot with a horizontal mean line and a shaded 95% CI band; right panel, two-bar comparison of the single-split validation R² vs the 5-fold CV mean, with a 95% CI error bar on the CV bar only."
>
> **After running, verify:**
> - Single-split validation R² is printed as one number
> - Five fold R² values are printed as an array
> - Mean, SD, and the two CI endpoints are printed with 4-decimal precision
> - Left panel: 5 bars with a red dashed mean line and a shaded CI band
> - Right panel: 2 bars — one for the single validation R², one for the 5-fold CV mean with a vertical error bar
> - Test set remains untouched
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# --- 1. Load California Housing and do the same 60/20/20 split used across NB01-NB05 ---
california = fetch_california_housing(as_frame=True)
X_reg = california.data
y_reg = california.target

X_reg_temp, X_reg_test, y_reg_temp, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_reg_train, X_reg_val, y_reg_train, y_reg_val = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
print(f"Train: {len(X_reg_train)} | Val: {len(X_reg_val)} | Test: {len(X_reg_test)} (locked)")

# --- 2. Fit Ridge on train, score once on the single validation split ---
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1.0, random_state=RANDOM_SEED))
])
ridge_pipeline.fit(X_reg_train, y_reg_train)
val_r2 = ridge_pipeline.score(X_reg_val, y_reg_val)
print(f"\nSingle-split validation R²: {val_r2:.4f}")

# --- 3. Run 5-fold CV on the TRAINING portion only (test set stays locked) ---
k = 5
cv = KFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(ridge_pipeline, X_reg_train, y_reg_train, cv=cv, scoring='r2')

mean_r2 = cv_scores.mean()
sd_r2 = cv_scores.std(ddof=1)
t_crit = stats.t.ppf(0.975, df=k - 1)
half_w = t_crit * sd_r2 / np.sqrt(k)
ci_low, ci_high = mean_r2 - half_w, mean_r2 + half_w

print(f"\n=== 5-FOLD CV (Ridge on California Housing) ===")
print(f"Fold R²:   {np.round(cv_scores, 4)}")
print(f"Mean R²:   {mean_r2:.4f}")
print(f"Std (k-1): {sd_r2:.4f}")
print(f"95% CI:    [{ci_low:.4f}, {ci_high:.4f}]  (half-width {half_w:.4f})")

# --- 4. Per-fold plot + Single-split vs CV comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-fold CV scores with mean line and CI band
axes[0].bar(range(1, k + 1), cv_scores, color='steelblue', edgecolor='black')
axes[0].axhline(mean_r2, color='red', linestyle='--', label=f'Mean = {mean_r2:.4f}')
axes[0].axhspan(ci_low, ci_high, color='red', alpha=0.15, label='95% CI')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('R² score')
axes[0].set_title('Per-fold 5-fold CV R² — Cal Housing, Ridge',
                  fontsize=12, fontweight='bold')
axes[0].legend(loc='lower right')

# Right: single-split validation vs CV mean with CI error bar
labels = ['Single validation split', '5-fold CV (mean)']
values = [val_r2, mean_r2]
errs = [0, half_w]
colors = ['goldenrod', 'steelblue']
bars = axes[1].bar(labels, values, yerr=errs, color=colors,
                   capsize=10, edgecolor='black')
axes[1].set_ylabel('R² score')
axes[1].set_title('Single Validation R² vs. 5-fold CV R² (with 95% CI)',
                  fontsize=12, fontweight='bold')
for i, v in enumerate(values):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# Quick status line for the CFO
inside = ci_low <= val_r2 <= ci_high
print(f"\n→ Single validation R² ({val_r2:.4f}) is "
      f"{'INSIDE' if inside else 'OUTSIDE'} the 5-fold CV 95% CI "
      f"[{ci_low:.4f}, {ci_high:.4f}].")

**Reading the output:**

The per-fold bar plot (left) shows the five $R^2$ scores you get by training Ridge on 80% of the training set and evaluating on the remaining 20% — repeated five times, with each tract landing in exactly one validation fold. On California Housing the fold scores usually land between **0.58 and 0.62**, a mean near **0.60**, a standard deviation around **0.01**, and a 95% CI half-width near **0.01** (because $k=5$ is small enough that the $t$ critical value pulls the half-width back up to \~1.24×SD).

The comparison bar plot (right) lines up the **single-split validation $R^2$** against the **5-fold CV mean with a 95% CI error bar**. Three outcomes are possible; the printed status line tells you which one you are in:

- **Inside the CI:** the single split is representative — the CFO can quote either number and the story is the same.
- **Above the CI:** the single split was *lucky* — the validation set happened to contain "easy" tracts. The honest estimate is the CV mean.
- **Below the CI:** the single split was *unlucky* — the validation set pulled in "hard" tracts. Again, quote the CV mean with its CI, not the one-split number.

**Interpretation:**

- **The CV mean is the number to put on the CFO's slide.** The single validation $R^2$ is one draw; the CV mean is an average of five.
- **The CI width is a proxy for how much faith the CFO can put in the mean.** A half-width of \~0.01 says "all five folds agreed — this number is stable." A wide CI would say "the estimate depends heavily on which 20% you held out" and would push you toward repeated CV or a larger training set.
- **The test set is still locked.** Both bars on the right come from the training population; the \~4,128 test tracts are reserved for the one-shot deployment estimate in NB14 or the final project.

---

## 3. Stratified K-Fold CV for Classification — Breast Cancer (MedScreen)

Same recipe as Section 2, applied to the MedScreen pipeline from NB06–NB07. One addition: when the target is categorical, **stratify** the folds so each fold preserves the positive-class prevalence of the full training set.

For the breast cancer training set (roughly 341 patients, 37% malignant / 63% benign), a plain random 5-fold split could produce one fold with 30% malignant and another with 45% — injecting noise the CV is *supposed to average out*, not add. `StratifiedKFold` enforces the 37/63 ratio in every fold.

Following NB07, the scoring metric is **ROC-AUC** (the threshold-independent headline for model selection). Plugging AUC into the $k$-fold estimator:

$$\text{CV}_{(5)}^{\text{AUC}} \;=\; \frac{1}{5} \sum_{i=1}^{5} \text{AUC}_i$$

where $\text{AUC}_i$ is the ROC-AUC on held-out fold $i$. The four-step recipe is unchanged: single-split validation score, 5-fold CV on training data only, mean + SD + 95% CI, two-panel bar plot with interpretation.

> 💡 **Gemini Prompt:** "Load the breast cancer dataset. Do a 60/20/20 train/val/test split with `random_state=RANDOM_SEED` and `stratify=y` (both calls). Build a logistic regression pipeline (StandardScaler + LogisticRegression with `max_iter=1000`), fit it on `X_train`, and compute the single-split validation ROC-AUC on `X_val` using `roc_auc_score` with `predict_proba(X_val)[:, 1]`. Then run 5-fold `StratifiedKFold` CV on the training set with `scoring='roc_auc'`. Compute the mean, sample SD (`ddof=1`), and 95% CI using `scipy.stats.t.ppf(0.975, df=k-1)`. Print everything. Render a 1x2 matplotlib figure: left panel per-fold CV bar plot with mean line and shaded 95% CI band; right panel two-bar comparison of single-split validation ROC-AUC vs the CV mean, with a 95% CI error bar on the CV bar only."
>
> **After running, verify:**
> - Single-split validation ROC-AUC is printed as one number (\~0.99)
> - Five fold AUC values are printed as an array
> - Mean, SD, and the two CI endpoints are printed with 4-decimal precision
> - Left panel: 5 bars with a red dashed mean line and a shaded CI band
> - Right panel: 2 bars — one for the single validation AUC, one for the 5-fold CV mean with a vertical error bar
> - Stratification preserved the 37/63 class balance in each fold (no crash, no undefined AUC)
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# --- 1. Load breast cancer data and apply the same stratified 60/20/20 split used in NB06/NB07 ---
bc = load_breast_cancer(as_frame=True)
X_clf = bc.data
y_clf = bc.target

X_clf_temp, X_clf_test, y_clf_temp, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_clf_train, X_clf_val, y_clf_train, y_clf_val = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
print(f"Train: {len(X_clf_train)} | Val: {len(X_clf_val)} | Test: {len(X_clf_test)} (locked)")

# --- 2. Fit logistic regression on train, score once on validation ---
log_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
])
log_pipeline.fit(X_clf_train, y_clf_train)
val_auc = roc_auc_score(y_clf_val, log_pipeline.predict_proba(X_clf_val)[:, 1])
print(f"\nSingle-split validation ROC-AUC: {val_auc:.4f}")

# --- 3. Run 5-fold STRATIFIED CV on the training set ---
k = 5
cv_strat = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)
cv_scores_clf = cross_val_score(log_pipeline, X_clf_train, y_clf_train,
                                cv=cv_strat, scoring='roc_auc')

mean_auc = cv_scores_clf.mean()
sd_auc = cv_scores_clf.std(ddof=1)
t_crit = stats.t.ppf(0.975, df=k - 1)
half_w_clf = t_crit * sd_auc / np.sqrt(k)
ci_low_clf, ci_high_clf = mean_auc - half_w_clf, mean_auc + half_w_clf

print(f"\n=== 5-FOLD STRATIFIED CV (LogReg on Breast Cancer) ===")
print(f"Fold AUC:  {np.round(cv_scores_clf, 4)}")
print(f"Mean AUC:  {mean_auc:.4f}")
print(f"Std (k-1): {sd_auc:.4f}")
print(f"95% CI:    [{ci_low_clf:.4f}, {ci_high_clf:.4f}]  (half-width {half_w_clf:.4f})")

# --- 4. Per-fold plot + Single-split vs CV comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, k + 1), cv_scores_clf, color='steelblue', edgecolor='black')
axes[0].axhline(mean_auc, color='red', linestyle='--', label=f'Mean = {mean_auc:.4f}')
axes[0].axhspan(ci_low_clf, ci_high_clf, color='red', alpha=0.15, label='95% CI')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Per-fold 5-fold Stratified CV ROC-AUC — Breast Cancer, LogReg',
                  fontsize=12, fontweight='bold')
axes[0].legend(loc='lower right')

labels = ['Single validation split', '5-fold CV (mean)']
values = [val_auc, mean_auc]
errs = [0, half_w_clf]
colors = ['goldenrod', 'steelblue']
axes[1].bar(labels, values, yerr=errs, color=colors, capsize=10, edgecolor='black')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Single Validation ROC-AUC vs. 5-fold CV (with 95% CI)',
                  fontsize=12, fontweight='bold')
for i, v in enumerate(values):
    axes[1].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

inside_clf = ci_low_clf <= val_auc <= ci_high_clf
print(f"\n→ Single validation ROC-AUC ({val_auc:.4f}) is "
      f"{'INSIDE' if inside_clf else 'OUTSIDE'} the 5-fold CV 95% CI "
      f"[{ci_low_clf:.4f}, {ci_high_clf:.4f}].")

**Reading the output:**

The per-fold bar plot (left) shows five ROC-AUC values on held-out stratified folds of the breast cancer training data. On this dataset the fold AUCs usually land between **0.98 and 1.00**, a mean near **0.99**, and a standard deviation around **0.005–0.01**. With $k=5$ the 95% CI half-width is \~1.24 × SD, so the CI is tight.

The comparison bar plot (right) lines up the **single-split validation ROC-AUC** (from NB07's workflow) against the **5-fold CV mean with a 95% CI error bar**. The printed status line tells you whether the single-split number is inside or outside the CI:

- **Inside the CI:** the NB07 validation AUC was representative — the chief medical officer can quote either number.
- **Above the CI:** the single split was *lucky* — those 114 validation patients happened to be easy ones. The CV mean is the honest estimate.
- **Below the CI:** the single split was *unlucky* — the 114 patients were on the hard side. Again, quote the CV mean with its CI.

**Interpretation:**

- **The CV mean is the number to put on the chief medical officer's slide.** The single validation AUC is one draw; the CV mean is an average of five.
- **Stratification shows up in the CI width.** Because every fold preserves the 37/63 malignant/benign ratio, the per-fold AUCs are tightly clustered and the CI is narrow. If you had used plain `KFold` instead, you would see a wider CI, and in extreme cases a fold with zero malignant patients would have made AUC undefined.
- **Model-selection rule from NB07 applies here.** AUC is the headline for *which classifier ranks best*. The CV mean + CI is what makes that headline statistically defensible — precision and recall at a chosen threshold still come from a separate deployment-readiness report.
- **Test set stays locked.** Both bars on the right come from training-side data; the 114 test patients are untouched for the one-shot deployment estimate later in the course.

---

## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Does Ridge **convincingly** beat plain OLS on California Housing?

Run the same 5-fold CV recipe from Section 2 with a plain `LinearRegression` pipeline (no L2 penalty), keeping everything else identical — same `X_reg_train`, same `KFold` splitter with `random_state=RANDOM_SEED`, same `scoring='r2'`. Compute its mean $R^2$, sample SD, and 95% CI using the Student's $t$ formula from Section 1, then build a two-bar comparison plot with **Ridge** (from Section 2) and **LinearRegression** bars, each with its 95% CI error bar.

**Decision rule:** If the two 95% CIs **overlap**, the Ridge improvement from the L2 penalty is *not* statistically convincing on this dataset — the CFO would not be able to defend "we need regularization" to the board. If the CIs **do not overlap**, regularization is buying a real, measurable edge.

---

> 💡 **Gemini Prompt:** "Build a plain linear regression pipeline (StandardScaler + LinearRegression). Using the same `X_reg_train`, `y_reg_train`, and `KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` from Section 2, run `cross_val_score` with `scoring='r2'`. Compute the mean, sample SD with `ddof=1`, and a 95% CI via `scipy.stats.t.ppf(0.975, df=4) * sd / sqrt(5)`. Print all three numbers. Then render a bar plot with two bars — 'Ridge (α=1.0)' (reusing `mean_r2`, `half_w` from Section 2) and 'Plain OLS' — each with a 95% CI error bar. End with a single printed line that says whether the two CIs overlap."
>
> **After running, verify:**
> - OLS mean R², SD, and CI are printed with 4-decimal precision
> - Bar plot has exactly two bars with 95% CI error bars
> - Bars are labelled 'Ridge (α=1.0)' and 'Plain OLS'
> - A final line prints either "CIs OVERLAP — Ridge's improvement is not statistically convincing" or "CIs do NOT overlap — Ridge's improvement is real"
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Build a plain OLS pipeline: StandardScaler + LinearRegression
#   2. Run 5-fold CV on X_reg_train, y_reg_train with scoring='r2'
#      (use the same KFold object from Section 2)
#   3. Compute mean, sample SD (ddof=1), and 95% CI using Student's t
#   4. Plot a 2-bar comparison (Ridge vs OLS) with CI error bars
#   5. Print whether the two CIs overlap
#
# Variables available from Section 2:
#   - X_reg_train, y_reg_train
#   - cv (the 5-fold KFold splitter with random_state=RANDOM_SEED)
#   - mean_r2, sd_r2, half_w, ci_low, ci_high (Ridge results)


### YOUR ANALYSIS:

**Question 1:** Do the Ridge and OLS 95% CIs overlap?  
[Your answer — inspect the plot or compare the printed CI endpoints]

**Question 2:** What would you tell the CFO based on this result?  
[Your answer — "we should deploy Ridge because …" or "plain OLS is enough because …"]

**Question 3:** The two model's CIs could change if you used a different `k` (3-fold vs. 10-fold) or a different `scoring` (negative MSE). Which change would shrink the CI most, and why?  
[Your answer]

---

## 📝 PAUSE-AND-DO Exercise 2 (10 minutes)

**Task:** Does regularization **strength** change MedScreen's ROC-AUC in a way the Health Department would care about?

Section 3 ran 5-fold stratified CV with the default `LogisticRegression(C=1.0)`. Now rerun the same CV recipe on `X_clf_train`/`y_clf_train` with a **strongly regularized** classifier — `LogisticRegression(C=0.01)` — keeping the `StandardScaler`, the same `StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` splitter, and `scoring='roc_auc'` identical. Compute its mean ROC-AUC, sample SD, and 95% CI, then build a two-bar comparison plot with **LogReg (C=1.0)** (from Section 3) and **LogReg (C=0.01)** bars, each with its 95% CI error bar.

**Decision rule:** If the two 95% CIs **overlap**, the choice of regularization strength does **not** move MedScreen's screening performance in a statistically convincing way on this data — the Health Department can pick on interpretability or operational grounds. If the CIs **do not overlap**, one setting is a real, measurable winner on ROC-AUC.

This is a preview of NB09: `GridSearchCV` runs exactly this comparison across many values of `C` and picks the best.

---

> 💡 **Gemini Prompt:** "Build a logistic regression pipeline with StandardScaler + LogisticRegression(C=0.01, random_state=RANDOM_SEED, max_iter=5000). Using the same `X_clf_train`, `y_clf_train`, and `StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` from Section 3, run `cross_val_score` with `scoring='roc_auc'`. Compute the mean, sample SD with `ddof=1`, and a 95% CI via `scipy.stats.t.ppf(0.975, df=4) * sd / sqrt(5)`. Print all three numbers with 4-decimal precision. Then render a bar plot with two bars — 'LogReg (C=1.0)' (reusing `mean_auc` and `half_w_clf` from Section 3) and 'LogReg (C=0.01)' — each with a 95% CI error bar. End with a single printed line that says whether the two CIs overlap."
>
> **After running, verify:**
> - Strong-regularization mean ROC-AUC, SD, and CI are printed with 4-decimal precision
> - Bar plot has exactly two bars with 95% CI error bars
> - Bars are labelled 'LogReg (C=1.0)' and 'LogReg (C=0.01)'
> - A final line prints either "CIs OVERLAP — regularization strength does not convincingly change ROC-AUC" or "CIs do NOT overlap — one C value is a real winner"
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Build a strong-regularization pipeline: StandardScaler + LogisticRegression(C=0.01, max_iter=5000)
#   2. Run 5-fold STRATIFIED CV on X_clf_train, y_clf_train with scoring='roc_auc'
#      (use the same StratifiedKFold object from Section 3)
#   3. Compute mean, sample SD (ddof=1), and 95% CI using Student's t
#   4. Plot a 2-bar comparison (C=1.0 vs C=0.01) with CI error bars
#   5. Print whether the two CIs overlap
#
# Variables available from Section 3:
#   - X_clf_train, y_clf_train
#   - cv_strat (the 5-fold StratifiedKFold splitter with random_state=RANDOM_SEED)
#   - mean_auc, sd_auc, half_w_clf, ci_low_clf, ci_high_clf (C=1.0 results)


### YOUR ANALYSIS:

**Question 1:** Do the C=1.0 and C=0.01 95% CIs overlap?  
[Your answer — inspect the plot or compare the printed CI endpoints]

**Question 2:** Based on this result, should MedScreen adopt stronger regularization in production?  
[Your answer — use the CI-overlap verdict, not the point estimates]

**Question 3:** NB09 will automate this comparison over a whole grid of `C` values. What role does the CI you just computed play inside `GridSearchCV`?  
[Your answer]

---

## 4. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **One split is one draw from a distribution.** A single 60/20/20 split produces one number; $k$-fold CV produces $k$ numbers whose mean, standard deviation, and 95% confidence interval form a much more honest estimate of production performance.

2. **The CV equation is the same across problem types.** Regression averages per-fold MSE; classification averages per-fold misclassification rate (or a monotone-equivalent score like ROC-AUC, accuracy, or F1). Only two new ingredients for classification: **stratification** (preserve class prevalence in every fold) and a **score-based** loss that matches the business question from NB07.

3. **Always report three numbers and a picture.** The mean is the point estimate, the sample SD ($\text{ddof}=1$) measures fold-to-fold noise, and the 95% CI using Student's $t$ with $k-1$ degrees of freedom is the range the CFO or the chief medical officer actually cares about. Pair it with a per-fold bar plot so the distribution is visible.

4. **Compare single-split against CV mean.** If the validation bar sits inside the CV 95% CI, the single split was representative. If outside, the split was lucky or unlucky and the CV mean is the honest estimate. This rule carries forward into NB09 (tuning) and the final project's evaluation section.

### Critical Rules:

> **"Never trust a single split — always cross-validate before a model-choice decision."**

> **"Quote the CV mean with its 95% confidence interval, never the mean alone."**

> **"Stratify folds for classification so class balance survives every fold."**

### Next Steps:

- Next notebook (NB09): hyperparameter tuning with `GridSearchCV` — the CV estimator in this notebook is the **inner loop** that decides which hyperparameter configuration wins.
- Apply today's CV framework to your project dataset: compute both a single validation score and a 5-fold CV mean ± CI, and put the two-panel comparison plot in your project write-up.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* - Model Assessment and Selection (k-fold CV, resampling concepts)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* - Resampling theory and selection bias
- scikit-learn User Guide: [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- Kohavi, R. (1995). "A study of cross-validation and bootstrap for accuracy estimation and model selection." *IJCAI*.

---



<center>

Thank you!

</center>